<a href="https://colab.research.google.com/github/GuardinTheDev/Is-This-Text-Ai-/blob/Model-E%C4%9Fitimi/ModelE%C4%9Fitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset, concatenate_datasets

print("Hugging Face Hub'dan 'Yunij/kaggle-comp-daigt' yükleniyor...")
# Orijinal veri setinin sadece 'train' kısmını alıyoruz (Birleştirme yapabilmek için)
raw_dataset = load_dataset('Yunij/kaggle-comp-daigt', split='train')

# =====================================================================
# 💉 RESMİ İNSAN METİNLERİ (WIKIPEDIA) ENJEKSİYONU
# =====================================================================
print("Dengeleme için resmi insan makaleleri (Wikipedia) indiriliyor...")
# Wikipedia makalelerini içeren popüler bir dil veri setini çekiyoruz
wiki_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')

# Modelin kafasını karıştıracak kısa başlıkları atalım, sadece 50 kelimeden uzun, gerçek makale paragraflarını alalım
wiki_filtered = wiki_dataset.filter(lambda x: len(x['text'].split()) > 50)

# Veri setinden 3000 adet ciddi insan makalesi alıyoruz (Eğer istersen bu sayıyı 5000 falan yapabilirsin)
wiki_samples = wiki_filtered.select(range(10000))

# Orijinal veri setindeki etiket sütununun adını dinamik bulalım ('label' veya 'generated' olabilir)
orijinal_sutunlar = raw_dataset.column_names
label_col = 'label' if 'label' in orijinal_sutunlar else 'generated'
text_col = 'text'

# Wikipedia metinlerini, senin veri setinle aynı formata sokuyoruz ve hepsine İNSAN (0) etiketini basıyoruz
def format_wiki(example):
    return {text_col: example['text'], label_col: 0}

wiki_formatted = wiki_samples.map(format_wiki, remove_columns=wiki_samples.column_names)

# Birleşmede hata çıkmaması için orijinal veri setindeki gereksiz/ekstra sütunları siliyoruz
sutunlari_tut = [text_col, label_col]
raw_dataset = raw_dataset.remove_columns([col for col in orijinal_sutunlar if col not in sutunlari_tut])

# Orijinal Kaggle verisi ile yeni Wikipedia makalelerini BİRLEŞTİRİYORUZ
print("Orijinal veri seti ile yeni resmi makaleler harmanlanıyor...")
combined_dataset = concatenate_datasets([raw_dataset, wiki_formatted])

# Model sırayla hep aynı şeyleri görmesin diye verileri iyice karıştırıyoruz
combined_dataset = combined_dataset.shuffle(seed=42)
# =====================================================================

# Eğitim ve Doğrulama (Validation) olarak bölelim
print("Eğitim ve doğrulama setleri ayrılıyor...")
split_data = combined_dataset.train_test_split(test_size=0.2, seed=42)
dataset = {
    'train': split_data['train'],
    'validation': split_data['test']
}

print("\n✅ Veri Seti Başarıyla Güncellendi, Dengelendi ve Bölümleri Ayarlandı!")
print("Eğitim seti boyutu:", len(dataset['train']))
print("Doğrulama seti boyutu:", len(dataset['validation']))
print("\nÖrnek bir veri (eğitim setinden):", dataset['train'][0])


In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# Model altyapısını hazır veri setine bağlama adımı
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fonksiyonu(examples):
    # DİKKAT: padding='max_length' kısmını sildik!
    # Sadece 512'den uzunları kesiyoruz (truncation).
    return tokenizer(examples['text'], truncation=True, max_length=512)

# Hafızadaki yerel veri setimizi tokenlaştırıyoruz
tokenized_datasets = {
    'train': dataset['train'].map(tokenize_fonksiyonu, batched=True),
    'validation': dataset['validation'].map(tokenize_fonksiyonu, batched=True)
}

# 🚀 DİNAMİK PADDING İÇİN DATA COLLATOR (Eğitimi Hızlandırır)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenlaştırma tamamlandı, model eğitimine hazır!")

# ***Model Eğitim Bloğu***

In [ ]:
import torch
import numpy as np
import os
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# --- KONTROL AYARI ---
# True yapılırsa dosya olsa bile eğitimi başlatır.
YENIDEN_EGIT = False
# ---------------------

yerel_kayit_yolu = "/content/en_iyi_detektor_modeli"

# 🚀 ÇÖZÜM BURADA: Eğitim kontrolünden ÖNCE modeli Drive linkinden indiriyoruz
if not os.path.exists(yerel_kayit_yolu) and not YENIDEN_EGIT:
    print("Model dosyaları buluttan otomatik indiriliyor, lütfen bekleyin...")
    klasor_id = "1XAhH433yk0KZanWj8ZH2mzPx01Ow0meO"
    !pip install -q gdown
    !gdown --folder https://drive.google.com/drive/folders/{klasor_id} -O {yerel_kayit_yolu}

model_mevcut = os.path.exists(yerel_kayit_yolu)

if model_mevcut and not YENIDEN_EGIT:
    print(f"✅ Model zaten '{yerel_kayit_yolu}' klasöründe mevcut ve YENIDEN_EGIT=False. Eğitim atlanıyor.")
else:
    if YENIDEN_EGIT:
        print("🔄 YENIDEN_EGIT=True olduğu için eğitim zorlanıyor...")
    else:
        print("⚠️ Model buluttan indirilemediği için mecburen eğitim başlıyor...")

    # 2. Cihaz Kontrolü
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Eğitim için kullanılan cihaz: {device}")

    # 3. Modeli Sınıflandırma İçin Yüklüyoruz
    model_name = "distilbert-base-uncased"
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    # 4. Model Başarısını Ölçmek İçin Metrik Fonksiyonu
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        acc = accuracy_score(labels, predictions)
        f1 = f1_score(labels, predictions, average='binary')
        return {"accuracy": acc, "f1": f1}

    # 5. Eğitim Hiperparametreleri
    training_args = TrainingArguments(
        output_dir="./ai_detector_local_results",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_steps=1,
        report_to="none"
    )

    # 6. Trainer Kurulumu
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['validation'],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )

    # 7. Eğitimi Başlatıyoruz
    print("\n--- Model Eğitimi Başlıyor ---")
    trainer.train()

    # 8. Modeli Kaydetme
    trainer.save_model(yerel_kayit_yolu)
    tokenizer.save_pretrained(yerel_kayit_yolu)
    print(f"\nEğitim tamamlandı ve model Colab yerel hafızasına kaydedildi: {yerel_kayit_yolu}")

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# MODELİ SAKİNLEŞTİRME AYARLARI
# ==========================================
AI_ESIK = 70.0     # Modelin AI demesi için çıtayı iyice yukarı çekiyoruz.
SICAKLIK = 3.5     # Değer büyüdükçe (örn: 2.0 - 3.5) modelin %99'luk inatçılığı kırılır,
                   # yüzdeler daha insani seviyelere (%60-%70) çekilir.

model_path = "/content/en_iyi_detektor_modeli"

if not os.path.exists(model_path):
    print(f"HATA: '{model_path}' klasörü bulunamadı! Lütfen üstteki eğitim/indirme hücresini çalıştırın.")
else:
    print("Model hafızaya yükleniyor...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()
    print("Model başarıyla yüklendi!\n")

    def metni_analiz_et(metin, threshold=95.0, temperature=2.5):
        inputs = tokenizer(metin, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Ham puanları (logits) sıcaklık değerine bölerek yumuşatıyoruz.
        yumusak_logits = outputs.logits / temperature
        olasiliklar = torch.softmax(yumusak_logits, dim=-1)[0]

        insan_skoru = olasiliklar[0].item() * 100
        yz_skoru = olasiliklar[1].item() * 100

        # Karar mekanizması yeni eşiğe göre çalışıyor
        if yz_skoru >= threshold:
            karar = "Yapay Zeka Tarafından Yazılmış 🤖"
        else:
            karar = "İnsan Tarafından Yazılmış 👨‍💻"

        return karar, insan_skoru, yz_skoru

    print("=== ANALİZ FONKSİYONU HAZIR ===")
    print("Artık Gradio arayüzünü kullanarak test yapabilirsiniz.")

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr

def gradio_analiz(metin):
    if not metin.strip():
        return "Lütfen bir metin girin.", 0, 0

    # Existing analysis function logic
    karar, insan_yuzde, yz_yuzde = metni_analiz_et(metin, threshold=AI_ESIK, temperature=SICAKLIK)

    return karar, f"%{insan_yuzde:.2f}", f"%{yz_yuzde:.2f}"

# Gradio Arayüz Tasarımı
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Yapay Zeka Metin Detektörü")
    gr.Markdown("Metnin yapay zeka tarafından mı yoksa insan tarafından mı yazıldığını analiz edin.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label="Analiz Edilecek Metin", placeholder="Metni buraya yapıştırın...", lines=10)
            submit_btn = gr.Button("Analiz Et", variant="primary")

        with gr.Column():
            label_output = gr.Label(label="Karar")
            with gr.Row():
                insan_output = gr.Textbox(label="İnsan Tahmini")
                yz_output = gr.Textbox(label="Yapay Zeka Tahmini")

    submit_btn.click(
        fn=gradio_analiz,
        inputs=input_text,
        outputs=[label_output, insan_output, yz_output]
    )

demo.launch(share=True, debug=True)